# Intel Image Classification — Transfer Learning (MobileNetV2)
**Task:** Multi-class Image Classification (6 classes)  
**Model:** MobileNetV2 backbone + custom head, built **manually** — no drag-and-drop fine-tuning  
**Training Strategy:** Phase 1 → freeze backbone; Phase 2 → unfreeze last 30 layers

## 1. Imports & Configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.preprocessing import label_binarize

print(f'TensorFlow: {tf.__version__}')

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

BASE_DIR    = '../data'
TRAIN_DIR   = os.path.join(BASE_DIR, 'train')
VAL_DIR     = os.path.join(BASE_DIR, 'val')
TEST_DIR    = os.path.join(BASE_DIR, 'test')
RESULTS_DIR = '../results'
MODELS_DIR  = '../models'

IMG_SIZE    = 224    # MobileNetV2 preferred input size
BATCH_SIZE  = 32
NUM_CLASSES = 6
CLASSES     = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

## 2. Data Generators (MobileNetV2 preprocessing)

In [ ]:
# MobileNetV2 expects pixel values in [-1, 1] via preprocess_input
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # normalise to [-1, 1]
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    seed=SEED, shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    seed=SEED, shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print(f'Train: {train_gen.n} | Val: {val_gen.n} | Test: {test_gen.n}')

## 3. Build Transfer Learning Model — Manually (no drag-and-drop)

In [ ]:
# ── Step 1: Load backbone WITHOUT top (classification head) ──────────────────
backbone = MobileNetV2(
    weights='imagenet',          # pre-trained on ImageNet
    include_top=False,           # exclude the original Dense head
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# ── Step 2: Freeze ALL backbone layers (Phase 1) ─────────────────────────────
backbone.trainable = False
print(f'Backbone layers : {len(backbone.layers)}')
print(f'Trainable layers after freezing: {sum(1 for l in backbone.layers if l.trainable)}')

# ── Step 3: Build model by explicitly attaching a custom head ─────────────────
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input')

# Pass through frozen backbone
x = backbone(inputs, training=False)          # training=False → BN layers stay in inference mode

# Global average pooling to flatten spatial dimensions
x = layers.GlobalAveragePooling2D(name='gap')(x)

# Custom Dense head — every layer explicitly defined
x = layers.Dense(512, name='fc1')(x)
x = layers.BatchNormalization(name='bn_fc1')(x)
x = layers.Activation('relu', name='relu_fc1')(x)
x = layers.Dropout(0.40, name='drop_fc1')(x)

x = layers.Dense(256, name='fc2')(x)
x = layers.BatchNormalization(name='bn_fc2')(x)
x = layers.Activation('relu', name='relu_fc2')(x)
x = layers.Dropout(0.30, name='drop_fc2')(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

tl_model = keras.Model(inputs, outputs, name='TL_MobileNetV2')
tl_model.summary()

## 4. Phase 1 — Train Custom Head Only

In [ ]:
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint(
        os.path.join(MODELS_DIR, 'tl_phase1_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

start_p1 = time.time()
history_p1 = tl_model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    verbose=1
)
phase1_time = time.time() - start_p1
print(f'\nPhase 1 completed in {phase1_time/60:.1f} minutes')
print(f'Best val accuracy: {max(history_p1.history["val_accuracy"]):.4f}')

## 5. Phase 2 — Fine-tune Last 30 Backbone Layers

In [ ]:
# ── Unfreeze last 30 layers of the backbone ───────────────────────────────────
backbone.trainable = True

# Freeze all layers except the last 30
UNFREEZE_FROM = len(backbone.layers) - 30
for i, layer in enumerate(backbone.layers):
    layer.trainable = (i >= UNFREEZE_FROM)

trainable_now = sum(1 for l in backbone.layers if l.trainable)
print(f'Unfrozen backbone layers: {trainable_now} / {len(backbone.layers)}')
print('Total trainable params after unfreezing:')
trainable_params = sum([
    tf.size(v).numpy() for v in tl_model.trainable_variables
])
print(f'  {trainable_params:,}')

# Recompile with a much lower learning rate for fine-tuning
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # 100× lower LR
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint(
        os.path.join(MODELS_DIR, 'tl_phase2_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

start_p2 = time.time()
history_p2 = tl_model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    verbose=1
)
phase2_time = time.time() - start_p2
tl_total_time = phase1_time + phase2_time

print(f'\nPhase 2 completed in {phase2_time/60:.1f} minutes')
print(f'Total TL training time: {tl_total_time/60:.1f} minutes')

In [ ]:
# ── Combined training curves (Phase 1 + Phase 2) ──────────────────────────────
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

combined = merge_histories(history_p1, history_p2)
phase1_end = len(history_p1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transfer Learning (MobileNetV2) — Training History', fontsize=14, fontweight='bold')

for ax, metric, label in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(combined[metric],     label=f'Train {label}',  color='#4E79A7', lw=2)
    ax.plot(combined[f'val_{metric}'], label=f'Val {label}', color='#E15759', lw=2, ls='--')
    ax.axvline(x=phase1_end, color='green', ls=':', lw=2, label='Fine-tuning start')
    ax.set_xlabel('Epoch'); ax.set_ylabel(label)
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tl_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Evaluation on Test Set

In [ ]:
test_gen.reset()
y_pred_proba_tl = tl_model.predict(test_gen, verbose=1)
y_pred_tl       = np.argmax(y_pred_proba_tl, axis=1)
y_true          = test_gen.classes

acc_tl  = accuracy_score(y_true, y_pred_tl)
prec_tl = precision_score(y_true, y_pred_tl, average='weighted')
rec_tl  = recall_score(y_true, y_pred_tl, average='weighted')
f1_tl   = f1_score(y_true, y_pred_tl, average='weighted')

print('=' * 55)
print('   Transfer Learning (MobileNetV2) — Test Metrics')
print('=' * 55)
print(f'  Accuracy  : {acc_tl:.4f}  ({acc_tl*100:.2f}%)')
print(f'  Precision : {prec_tl:.4f}')
print(f'  Recall    : {rec_tl:.4f}')
print(f'  F1-Score  : {f1_tl:.4f}')
print('=' * 55)

tl_metrics = {
    'accuracy': acc_tl, 'precision': prec_tl, 'recall': rec_tl, 'f1': f1_tl,
    'train_time_min': tl_total_time / 60,
    'trainable_params': int(sum(tf.size(v).numpy() for v in tl_model.trainable_variables))
}
with open(os.path.join(RESULTS_DIR, 'tl_metrics.json'), 'w') as fp:
    json.dump(tl_metrics, fp, indent=2)

In [ ]:
print('Classification Report:\n')
print(classification_report(y_true, y_pred_tl, target_names=CLASSES))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm_tl = confusion_matrix(y_true, y_pred_tl)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_tl, annot=True, fmt='d', cmap='Greens',
    xticklabels=CLASSES, yticklabels=CLASSES,
    linewidths=0.5, ax=ax
)
ax.set_title('Confusion Matrix — Transfer Learning (MobileNetV2)', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tl_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
colors_roc = ['#4E79A7','#F28E2B','#E15759','#76B7B2','#59A14F','#EDC948']

fig, ax = plt.subplots(figsize=(10, 7))
for i, (cls, color) in enumerate(zip(CLASSES, colors_roc)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba_tl[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Transfer Learning (One-vs-Rest)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tl_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Comparison Table

In [ ]:
# Load scratch metrics
with open(os.path.join(RESULTS_DIR, 'scratch_metrics.json')) as fp:
    s = json.load(fp)

with open(os.path.join(RESULTS_DIR, 'tl_metrics.json')) as fp:
    t = json.load(fp)

print('\n' + '='*70)
print(f'{"Metric":<30} {"CNN Scratch":>18} {"MobileNetV2 TL":>18}')
print('='*70)
metrics_compare = [
    ('Test Accuracy',       f"{s['accuracy']*100:.2f}%",          f"{t['accuracy']*100:.2f}%"),
    ('Precision (weighted)',f"{s['precision']:.4f}",               f"{t['precision']:.4f}"),
    ('Recall (weighted)',   f"{s['recall']:.4f}",                  f"{t['recall']:.4f}"),
    ('F1-Score (weighted)', f"{s['f1']:.4f}",                      f"{t['f1']:.4f}"),
    ('Training Time (min)', f"{s['train_time_min']:.1f}",          f"{t['train_time_min']:.1f}"),
    ('Trainable Params',    f"{s['trainable_params']:,}",          f"{t['trainable_params']:,}"),
]
for m, sv, tv in metrics_compare:
    print(f'{m:<30} {sv:>18} {tv:>18}')
print('='*70)

In [ ]:
# ── Visual comparison bar chart ────────────────────────────────────────────────
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
scratch_vals  = [s['accuracy'], s['precision'], s['recall'], s['f1']]
tl_vals       = [t['accuracy'], t['precision'], t['recall'], t['f1']]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, scratch_vals, width, label='CNN from Scratch', color='#4E79A7')
bars2 = ax.bar(x + width/2, tl_vals,     width, label='MobileNetV2 TL',   color='#59A14F')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Model Comparison — CNN Scratch vs Transfer Learning', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save final model
tl_model.save(os.path.join(MODELS_DIR, 'tl_mobilenetv2_final.keras'))
print('Transfer Learning model saved!')